### Aula: Implementando Ensemble Learning usando sklearn


### 1. Introdução
- Nesta aula, vamos explorar a implementação de diferentes técnicas de **Ensemble Learning** usando a biblioteca sklearn.
- Focaremos em problemas de **classificação** e abordaremos as seguintes técnicas:
    - Voting
    - Bagging
    - Boosting
    - Stacking

### 2. Configuração do Ambiente

Primeiro, precisamos importar as bibliotecas comuns as técnicas:

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### 3. Criação do conjunto de dados
- Usaremos a função ``make_classification()`` para criar um conjunto de dados de classificação binária de teste
- O conjunto de dados terá 1.000 exemplos, com 20 características (atributos) de entrada
- Vamos fixar a semente do gerador de números aleatórios para garantir que obtenhamos os mesmos exemplos a cada vez que o código for executado

In [ ]:
# Gerando um conjunto de dados de classificação
X, y = make_classification(n_samples=1000,
                               n_features=20, random_state=42)

- **Divisão estratificada em treino e teste do conjunto de dados**
    - O parâmetro ``test_size=0.2`` define a proporção dos dados que serão utilizados como conjunto de teste (neste caso, 20%)
    - o parâmetro ``stratify=y`` garante que a divisão seja estratificada, ou seja, as proporções de classes em $y$ sejam mantidas nos conjuntos de treinamento e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                            random_state=42, stratify=y)

### 4. Definindo uma lista de modelos de ML para avaliar
- Será avaliado um conjunto de diferentes modelos de ML
- Especificamente, será avaliado os seguintes algoritmos:
    - Regressão Logística
    - k-Nearest Neighbors (k-Vizinhos Mais Próximos)
    - Árvore de Decisão
    - Naive Bayes
- Cada algoritmo será avaliado utilizando os hiperparâmetros padrão do modelo

In [ ]:
#Importação das classes de cada Algoritmo de ML implementada no sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Definindo uma lista de modelos para avaliar
base_models = [
    ('lr', LogisticRegression(random_state=42)),
    ('nb', GaussianNB()),
    ('knn', KNeighborsClassifier()),
    ('dt', DecisionTreeClassifier(random_state=42))
]

- **Avaliando individualmente cada modelo**

In [ ]:
# Avaliar Acurácia dos modelos individualmente no conjunto de teste
for name, model in base_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'{name} Classifier:')
    print(f'Acurácia: {accuracy_score(y_test, y_pred)*100:.2f}%')
    print("-" * 60)

### 5. Classificador de votação
- A técnica de Voting combina as previsões de vários modelos para determinar a classe final
- Pode ser feito por votação de maioria ou média ponderada das probabilidades

- **Definindo e treinando o classificador de Votação**

In [ ]:
from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(estimators=base_models,
                             voting='soft')
voting_clf.fit(X_train, y_train)

- **Avaliando do classificador de votação**

In [ ]:
# Resultados do Voting Classifier
y_pred = voting_clf.predict(X_test)
voting_clf_acc = accuracy_score(y_test, y_pred)
print("Classificador de Votação:")
print(f'Accuracy: {voting_clf_acc*100:.2f}%')

### 6. Bagging Classifier
- Consiste em treinar diversos modelos de forma independente e depois combinar suas previsões:
    - Bagging (Bootstrap Aggregating):
        - Utiliza amostragem com reposição para criar conjuntos de treinamento para cada modelo.
        - Isso significa que os mesmos exemplos podem aparecer várias vezes em uma amostra.
    - Pasting:
        - Similar ao Bagging, mas utiliza amostragem sem reposição, ou seja, os exemplos são selecionados apenas uma vez para cada amostra.
- Sklearn disponibliza uma API simples para o bagging e para o pasting por intermédio da classe ``BaggingClassifier``
- Principais parâmetros da classe ``BaggingClassifier``
    - ``estimator``: O estimador base para ajustar em subconjuntos aleatórios do conjunto de dados. Se for None, então o estimador base é um ``DecisionTreeClassifier``

    - ``n_estimators``: Número de estimadores a serem utilizados no ensemble. Quanto maior o número de estimadores, mais robusto e geralmente melhor é o desempenho, mas isso também aumenta o custo computacional.

    - ``max_samples``: Número ou proporção de amostras a serem selecionadas aleatoriamente para treinar cada estimador. Se for um inteiro, representa o número exato de amostras. Se for um float, representa a proporção de amostras em relação ao tamanho do conjunto de treinamento.

    - ``bootstrap``: Indica se a amostragem deve ser realizada com ou sem substituição. Se for True, será feita uma amostragem com substituição (Bagging). Se for False, será feita uma amostragem sem substituição (Pasting).
    - ``n_jobs``: O número de trabalhos a serem executados em paralelo para ``fit`` e ``predict``. Se ``None`` significa 1 e treinamento ocorrerá de forma sequencial, sem paralelismo. Quando definido como -1, utiliza todos os processadores disponíveis no computador.

 - **Criando e treinando o classificador Bagging**

In [ ]:
from sklearn.ensemble import BaggingClassifier
bag_clf = BaggingClassifier(DecisionTreeClassifier(random_state=42),
                            bootstrap=True, n_estimators=100,
                            max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

- O **Random Forest** é uma técnica específica de Bagging que utiliza árvores de decisão como aprendizes fracos
- Em vez de criar um ``BagginClassifier`` e passá-lo em um ``DecisionTreeClassifier``, pode-se usar a classe ``RandomForestClassifier``, que é otimizada para Árvores de Decisão

- **Avaliando o desempenho do classificador Bagging**

In [ ]:
# Resultados do Classificador Bagging
y_pred = bag_clf.predict(X_test)
bag_clf_acc = accuracy_score(y_test, y_pred)
print("Classificador Bagging:")
print(f'Accuracy: {bag_clf_acc*100:.2f}%')

### 7. Classificador Boosting
- Boosting ajusta novos modelos para corrigir os erros dos modelos anteriores.
- Aqui, abordaremos AdaBoost e Gradiente Boosting

In [ ]:
#Importação das Classes
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

### 7.1 AdaBoost
- Sklearn oferece uma implementação do algoritmo AdaBoost adaptada para problemas de classificação multiclasse, chamada SAMME [1]
- SAMME significa <i> Stagewise Additive Modeling using a Multiclass Exponential loss function</i> - em português Modelagem Aditiva Estagiada usando uma função de perda Exponencial Multiclasse
- Nessa implementação vamos treinar um classificador AdaBoost com base em 100 Decision Strumps usando a classe ``AdaBoostClassifier``

  [1] Ji Zhu et al. <b>Multi-Class Adaboost</b>. Statistics and Its interface 2, no 3 (2009): 349-360. Disponível em https://dept.stat.lsa.umich.edu/~jizhu/pubs/Zhu-SII09.pdf

 - **Criando e treinando o AdaBoost**

In [ ]:
# Criação e treinamento do AdaBoost Classifier
adaboost_clf = AdaBoostClassifier(DecisionTreeClassifier(), n_estimators=100,
                             random_state=42, algorithm='SAMME')
adaboost_clf.fit(X_train, y_train)

- **Avaliando o desempenho do classificador AdaBoost**

In [ ]:
# Resultados do Adaboost
y_pred = adaboost_clf.predict(X_test)
adaboost_clf_acc = accuracy_score(y_test, y_pred)
print("Classificador AdaBoost:")
print(f'Acurácia: {adaboost_clf_acc*100:.2f}%')

### 7.2 Gradiente Boosting
- Vamos usar a classe ``GradientBoostingClassifier`` da biblioteca scikit-learn
- Em cada estágio, árvores de regressão com n_classes_ são ajustadas ao gradiente da função de perda

 - **Criando e treinando o Gradiente Boosting**

In [ ]:
# Criação e treinamento do Gradiente Boosting
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
# Treinando o modelo
gb_clf.fit(X_train, y_train)

- **Avaliando o desempenho do classificador Gradiente Boosting**

In [ ]:
# Resultados do Gradiente Boosting
y_pred = gb_clf.predict(X_test)
gb_clf_acc = accuracy_score(y_test, y_pred)
print("Classificador Gradiente Boosting:")
print(f'Accuracy: {gb_clf_acc*100:.2f}%')

### 8. Stacking
- Stacking envolve treinar vários modelos (chamados de base models) e depois usar um modelo meta (meta-model) para combinar suas previsões

In [ ]:
from sklearn.ensemble import StackingClassifier

- **Criando e treinando o stacking**

In [ ]:
#Vamos utilizar o base-models criado

# Criação do meta-model
meta_model = LogisticRegression()

# Criação do Stacking Classifier
stacking_clf = StackingClassifier(estimators=base_models,
                                  final_estimator=meta_model, cv=10)

# Treinamento e avaliação
stacking_clf.fit(X_train, y_train)

- **Avaliando o desempenho do Stacking**

In [ ]:
# Resultados do Gradiente Boosting
y_pred = stacking_clf.predict(X_test)
stacking_clf_acc = accuracy_score(y_test, y_pred)
print("Classificador Stacking:")
print(f'Acurácia: {stacking_clf_acc*100:.2f}%')

### 9. Resumo dos resultados

In [ ]:
# Avaliar Acurácia dos modelos individualmente no conjunto de teste
for name, model in base_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'{name} -> Acurácia: {accuracy_score(y_test, y_pred)*100:.2f}%')

print(f'Voting -> Acurácia: {voting_clf_acc*100:.2f}%')
print(f'Bagging -> Acurácia: {bag_clf_acc*100:.2f}%')
print(f'AdaBoost -> Acurácia: {adaboost_clf_acc*100:.2f}%')
print(f'Gradiente Boosting -> Acurácia: {gb_clf_acc*100:.2f}%')
print(f'Stacking -> Acurácia: {stacking_clf_acc*100:.2f}%')

### Considerações finais
- A tabela abaixo resume as vantagens e desvantagens das diferentes técnicas de ensemble learning que abordamos: Voting, Bagging, AdaBoost, Gradient Boosting e Stacking.

| Técnica              | Vantagens                                                                                  | Desvantagens                                                                                                 |
|----------------------|--------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------|
| **Voting Classifier**| - Simples de implementar e entender.                                                       | - Pode não melhorar significativamente a performance se os modelos base não forem diversificados.           |
|                      | - Pode combinar qualquer tipo de modelo.                                                   | - Depende fortemente da qualidade dos modelos base.                                                         |
| **Bagging**          | - Reduz a variância e ajuda a prevenir overfitting.                                         | - Não reduz o viés, portanto, não é tão eficaz se o modelo base já tiver baixo viés.                        |
|                      | - Funciona bem com modelos de alta variância, como árvores de decisão.                     | - Pode ser computacionalmente caro devido ao treinamento de múltiplos modelos.                              |
| **AdaBoost**         | - Frequentemente aumenta a precisão do modelo.                                             | - Sensível a outliers e dados ruidosos.                                                                     |
|                      | - Foca em exemplos difíceis, melhorando a performance em datasets complexos.               | - Pode sofrer de overfitting se não for bem regulado.                                                       |
| **Gradient Boosting**| - Excelente desempenho em termos de acurácia.                                              | - Computacionalmente intensivo e pode ser lento para treinar.                                               |
|                      | - Pode capturar relações complexas nos dados.                                              | - Propenso a overfitting se o número de modelos base for muito grande.                                      |
| **Stacking**         | - Pode capturar a força de múltiplos modelos base.                                          | - Complexo de implementar e treinar.                                                                        |
|                      | - Flexível, pois pode usar qualquer tipo de modelo base e meta-modelo.                     | - Computacionalmente caro e pode ser propenso a overfitting se não for bem regulado.                        |